# Outside water condensation (v0.6.0)

Partial H2O condensation from a wet process gas flowing across a bank of
bare tubes, cooled by dry air flowing inside the tubes.

See `core/phase_change/` and `docs/property_models.md` for the underlying
model.

In [1]:
import sys
from pathlib import Path

repository_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "core").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from core.geometry.bundle import TubeBundle
from core.geometry.tube import BareTube
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput
from core.properties.gas_mixture import GasMixtureSpec, GasMixturePropertyProvider
from core.phase_change.capability import detect_phase_change_capability
from core.phase_change.types import PhaseChangeMode

# Bare-tube bank geometry: 20 rows x 30 tubes/row, staggered layout.
tube = BareTube(D_i=25e-3 - 2 * 1.5e-3, D_o=25e-3, length_total=2.8, length_effective=2.8, wall_k=50.0)
bundle = TubeBundle(
    tube=tube, n_rows=20, n_tubes_per_row=30,
    pitch_transverse=35e-3, pitch_longitudinal=35e-3,
    layout="staggered", n_passes_tube=2, flow_arrangement="counterflow",
)
hx = BareTubeHeatExchanger(bundle)
hx.bundle.n_rows, hx.bundle.n_tubes_per_row

(20, 30)

## Wet process gas (outside) and its water content

In [2]:
wet_process_gas = GasMixtureSpec(
    components={"N2": 0.65, "O2": 0.10, "CO2": 0.08, "H2O": 0.17}, basis="mole",
)
outside_provider = GasMixturePropertyProvider(wet_process_gas)

outside_capability = detect_phase_change_capability(outside_provider)
print("phase-change capable:", outside_capability.capable)
print("W_in [kg H2O / kg dry gas]:", outside_capability.W_in)
print("dry-gas mole fractions:", outside_capability.dry_mole_fractions)
print("M_dry [kg/mol]:", outside_capability.M_dry)

phase-change capable: True
W_in [kg H2O / kg dry gas]: 0.12285108115534502
dry-gas mole fractions: {'Nitrogen': 0.783132530120482, 'Oxygen': 0.12048192771084339, 'CarbonDioxide': 0.0963855421686747}
M_dry [kg/mol]: 0.030035361445783136


## Dew point of the inlet gas

In [3]:
from core.phase_change.water_equilibrium import water_dew_point, water_partial_pressure, water_mole_fraction_from_ratio

p_outside = 101_325.0
y_h2o_in = water_mole_fraction_from_ratio(outside_capability.W_in, M_dry=outside_capability.M_dry, M_h2o=outside_capability.M_condensable)
p_h2o_in = water_partial_pressure(y_h2o_in, p_outside)
T_dew_in = water_dew_point(p_h2o_in)
print(f"y_H2O,in = {y_h2o_in:.4f}  p_H2O,in = {p_h2o_in:.1f} Pa  T_dew,in = {T_dew_in:.2f} K = {T_dew_in - 273.15:.2f} degC")

y_H2O,in = 0.1700  p_H2O,in = 17225.2 Pa  T_dew,in = 330.02 K = 56.87 degC


## Dry air (inside)

In [4]:
dry_air = GasMixtureSpec(components={"N2": 0.79, "O2": 0.21}, basis="mole")
inside_provider = GasMixturePropertyProvider(dry_air)

inside = HXSideInput(provider=inside_provider, m_dot=15.0, T_in=290.0, p=101_325.0)
outside = HXSideInput(provider=outside_provider, m_dot=6.0, T_in=420.0, p=p_outside)
inside.T_in, outside.T_in, outside.m_dot

(290.0, 420.0, 6.0)

## Case A — AUTO, no condensation (dry-enough gas stays above its own dew point)

In [5]:
low_humidity_gas = GasMixtureSpec(components={"N2": 0.70, "O2": 0.10, "CO2": 0.12, "H2O": 0.08}, basis="mole")
outside_dry_case = HXSideInput(provider=GasMixturePropertyProvider(low_humidity_gas), m_dot=8.0, T_in=450.0, p=p_outside)
inside_for_dry_case = HXSideInput(provider=inside_provider, m_dot=5.0, T_in=300.0, p=101_325.0)

result_no_condensation = hx.simulate(inside_for_dry_case, outside_dry_case)
pc_a = result_no_condensation.outside_phase_change
print("capable:", pc_a.capable, " possible:", pc_a.possible, " active:", pc_a.active)
print("m_dot_condensate [kg/s]:", pc_a.m_dot_condensate, " Q_latent [W]:", pc_a.Q_latent)

capable: True  possible: False  active: False
m_dot_condensate [kg/s]: 0.0  Q_latent [W]: 0.0


## Case B — AUTO, partial outside condensation (the main v0.6.0 case)

In [6]:
result_auto = hx.simulate(inside, outside)
pc_b = result_auto.outside_phase_change
print("converged:", result_auto.converged, " active:", pc_b.active, " iterations:", pc_b.iterations)
print(f"Q_total = {pc_b.Q_total/1e3:.2f} kW   Q_sensible = {pc_b.Q_sensible/1e3:.2f} kW   Q_latent = {pc_b.Q_latent/1e3:.2f} kW")
print(f"T_out_inside = {result_auto.T_out_inside - 273.15:.2f} degC   T_out_outside = {result_auto.T_out_outside - 273.15:.2f} degC")
print(f"W_in = {pc_b.W_in:.4f}   W_out = {pc_b.W_out:.4f}   m_dot_condensate = {pc_b.m_dot_condensate * 3600:.2f} kg/h")

converged: True  active: True  iterations: 24
Q_total = 639.52 kW   Q_sensible = 560.31 kW   Q_latent = 79.21 kW
T_out_inside = 58.92 degC   T_out_outside = 59.58 degC
W_in = 0.1229   W_out = 0.1166   m_dot_condensate = 120.41 kg/h


### Onset and partial wet-area diagnostics (v0.6.0 fix)

Onset is decided from the *minimum* estimated wall temperature
(`onset_wall_temperature`), not the mean -- the coldest point of the
surface can be below the dew point (`possible=True`) even while the
mean-averaged wall is still nominally warm. Once condensing, the wet
fraction of the surface is estimated linearly between the wall-temperature
extrema and only that fraction of the area (`wet_area`) is used for mass
transfer / latent heat; sensible heat transfer still uses the whole
`outside_total_area`.

In [7]:
print(f"T_dew (onset)          = {pc_b.dew_point_in - 273.15:.2f} degC")
print(f"T_wall_min (onset)     = {pc_b.onset_wall_temperature - 273.15:.2f} degC   [method: {pc_b.onset_temperature_method}]")
print(f"onset margin           = {pc_b.onset_margin_K:.2f} K  (T_dew - T_wall_min; positive = below dew point)")
print(f"possible / active / near_onset = {pc_b.possible} / {pc_b.active} / {pc_b.near_onset}")
print()
print(f"T_wall_min  (converged wet solve) = {pc_b.wall_temperature_min - 273.15:.2f} degC")
print(f"T_wall_mean (converged wet solve) = {pc_b.wall_temperature_mean - 273.15:.2f} degC")
print(f"T_wall_max  (converged wet solve) = {pc_b.wall_temperature_max - 273.15:.2f} degC")
print(f"wet_surface_fraction = {pc_b.wet_surface_fraction:.3f}   [method: {pc_b.wet_surface_fraction_method}]")
print(f"A_wet = {pc_b.wet_area:.2f} m2   of A_outside = {pc_b.outside_total_area:.2f} m2")
print(f"m_dot_condensate = {pc_b.m_dot_condensate * 3600:.2f} kg/h   Q_sensible = {pc_b.Q_sensible/1e3:.2f} kW   Q_latent = {pc_b.Q_latent/1e3:.2f} kW")

# This is the case the fix targets: the *onset* check uses the dry
# baseline's minimum estimated wall temperature (onset_wall_temperature),
# which is well below the dew point here -- a mean-based check on the same
# dry baseline would have looked comfortably above it and missed the
# condensation entirely (see docs/property_models.md section 15.5).
assert pc_b.onset_wall_temperature < pc_b.dew_point_in
assert pc_b.possible and pc_b.active
assert 0.0 < pc_b.wet_surface_fraction < 1.0

T_dew (onset)          = 56.87 degC
T_wall_min (onset)     = 27.25 degC   [method: wall_temperature_min_envelope_estimate]
onset margin           = 29.62 K  (T_dew - T_wall_min; positive = below dew point)
possible / active / near_onset = True / True / False

T_wall_min  (converged wet solve) = 43.23 degC
T_wall_mean (converged wet solve) = 55.64 degC
T_wall_max  (converged wet solve) = 64.74 degC
wet_surface_fraction = 0.613   [method: linear_wall_temperature_envelope]
A_wet = 80.87 m2   of A_outside = 131.95 m2
m_dot_condensate = 120.41 kg/h   Q_sensible = 560.31 kW   Q_latent = 79.21 kW


## Case C — same case, phase_change_mode=DISABLED (forced sensible-only)

In [8]:
outside_disabled = HXSideInput(provider=outside_provider, m_dot=6.0, T_in=420.0, p=p_outside, phase_change_mode=PhaseChangeMode.DISABLED)
result_disabled = hx.simulate(inside, outside_disabled)
pc_c = result_disabled.outside_phase_change
print("active:", pc_c.active, " possible:", pc_c.possible, " m_dot_condensate:", pc_c.m_dot_condensate)
for w in pc_c.warnings:
    print(f"[{w.severity}] {w.code}: {w.message}")

active: False  possible: True  m_dot_condensate: 0.0
[warning] PHASE_CHANGE_DISABLED_BUT_POSSIBLE: outside: phase_change_mode=DISABLED forced a sensible-only result, but the dry baseline's minimum estimated wall temperature runs below the dew point -- condensation would be thermodynamically possible here. This result is a forced single-phase approximation: it does not remove condensate from the stream, and the true heat duty and outlet temperature may differ.


## Comparison table: AUTO (condensing) vs DISABLED (forced sensible-only)

In [9]:
rows = [
    ("Q_total [kW]", pc_b.Q_total / 1e3, pc_c.Q_total / 1e3),
    ("Q_sensible [kW]", pc_b.Q_sensible / 1e3, pc_c.Q_sensible / 1e3),
    ("Q_latent [kW]", pc_b.Q_latent / 1e3, pc_c.Q_latent / 1e3),
    ("T_out_inside [degC]", result_auto.T_out_inside - 273.15, result_disabled.T_out_inside - 273.15),
    ("T_out_outside [degC]", result_auto.T_out_outside - 273.15, result_disabled.T_out_outside - 273.15),
    ("dew_point_in [degC]", pc_b.dew_point_in - 273.15, pc_c.dew_point_in - 273.15),
    ("onset_wall_temperature (Tmin) [degC]", pc_b.onset_wall_temperature - 273.15, pc_c.onset_wall_temperature - 273.15),
    ("onset_margin_K [K]", pc_b.onset_margin_K, pc_c.onset_margin_K),
    ("wall_temperature_mean [degC]", pc_b.wall_temperature_mean - 273.15, (pc_c.wall_temperature_mean - 273.15) if pc_c.wall_temperature_mean else float("nan")),
    ("W_in [kg/kg]", pc_b.W_in, pc_c.W_in),
    ("W_out [kg/kg]", pc_b.W_out, pc_c.W_out),
    ("m_dot_condensate [kg/h]", pc_b.m_dot_condensate * 3600, pc_c.m_dot_condensate * 3600),
    ("wet_surface_fraction [-]", pc_b.wet_surface_fraction, pc_c.wet_surface_fraction),
    ("A_wet [m2]", pc_b.wet_area, pc_c.wet_area),
    ("A_outside [m2]", pc_b.outside_total_area, pc_c.outside_total_area),
    ("alfa_outside_dry [W/m2K]", pc_b.alfa_dry, result_disabled.outside_alfa_mean),
    ("alfa_outside_effective [W/m2K]", pc_b.alfa_effective, None),
]
print(f"{'quantity':38s} {'AUTO (condensing)':>20s} {'DISABLED':>20s}")
for name, auto_val, disabled_val in rows:
    auto_s = f"{auto_val:.4g}" if auto_val is not None else "n/a"
    dis_s = f"{disabled_val:.4g}" if disabled_val is not None else "n/a"
    print(f"{name:38s} {auto_s:>20s} {dis_s:>20s}")

quantity                                  AUTO (condensing)             DISABLED
Q_total [kW]                                          639.5                    0
Q_sensible [kW]                                       560.3                    0
Q_latent [kW]                                         79.21                    0
T_out_inside [degC]                                   58.92                54.76
T_out_outside [degC]                                  59.58                60.32
dew_point_in [degC]                                   56.87                56.87
onset_wall_temperature (Tmin) [degC]                  27.25                27.25
onset_margin_K [K]                                    29.62                29.62
wall_temperature_mean [degC]                          55.64                52.51
W_in [kg/kg]                                         0.1229               0.1229
W_out [kg/kg]                                        0.1166               0.1229
m_dot_condensate [kg/h]     

## Mass balance check (Case B)

In [10]:
m_dot_water_vapor_in = pc_b.m_dot_water_vapor_in
m_dot_water_vapor_out = pc_b.m_dot_water_vapor_out
print(f"m_dot_water_vapor_in  = {m_dot_water_vapor_in * 3600:.3f} kg/h")
print(f"m_dot_water_vapor_out = {m_dot_water_vapor_out * 3600:.3f} kg/h")
print(f"m_dot_condensate      = {pc_b.m_dot_condensate * 3600:.3f} kg/h")
print(f"mass_balance_error    = {pc_b.mass_balance_error:.3e} kg/s")
assert abs(pc_b.mass_balance_error) < 1e-6

m_dot_water_vapor_in  = 2363.255 kg/h
m_dot_water_vapor_out = 2242.846 kg/h
m_dot_condensate      = 120.409 kg/h
mass_balance_error    = -1.210e-08 kg/s


## Energy balance check (Case B)

In [11]:
print(f"Q_sensible + Q_latent = {(pc_b.Q_sensible + pc_b.Q_latent)/1e3:.3f} kW")
print(f"Q_total               = {pc_b.Q_total/1e3:.3f} kW")
print(f"energy_balance_error  = {pc_b.energy_balance_error:.3e} W")
assert abs(pc_b.energy_balance_error) < 1e-6

Q_sensible + Q_latent = 639.518 kW
Q_total               = 639.518 kW
energy_balance_error  = 0.000e+00 W


## Warnings and modelling assumptions (Case B)

In [12]:
print("assumptions:")
for a in pc_b.assumptions:
    print(" -", a)
print()
print("warnings:")
for w in pc_b.warnings:
    print(f" [{w.severity}] {w.code}: {w.message}")

assumptions:
 - bulk_mean_property_evaluation_0d
 - fully_drained_liquid_condensate
 - lewis_number_chilton_colburn_analogy
 - dry_gas_composition_unchanged_by_condensation
 - wet_surface_fraction_two_point_inlet_outlet_estimate

warnings:
 [info] outside_dp_zukauskas_staggered_sl_over_d_offgrid: outside_dp: current open Zukauskas provider is strongest near staggered SL/D = 1.25, 1.5, 2.0 and uses interpolation/clamping outside these points.
 [info] WET_SURFACE_FRACTION_0D_ESTIMATE: outside: wet_surface_fraction is a 0D linear estimate (linear_wall_temperature_envelope) based on a cheap two-point (inlet/outlet) wall-temperature estimate, not a spatially resolved (1D/segmented) wetted-area fraction.
 [info] OUTSIDE_CONDENSATION_DETECTED: outside: the dry sensible-only baseline showed the outside tube-wall surface running below the water dew point; partial H2O condensation was solved for this call.
 [info] LEWIS_NUMBER_ASSUMED: outside: mass transfer used the Chilton-Colburn analogy with